## Load dataset

In [9]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")
df = dataset["train"].to_pandas()

'[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1016)' thrown while requesting HEAD https://huggingface.co/datasets/stanfordnlp/imdb/resolve/e6281661ce1c48d982bc483cf8a173c1bbeb5d31/imdb.py
Retrying in 1s [Retry 1/5].
Using the latest cached version of the dataset since stanfordnlp/imdb couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'plain_text' at C:\Users\lalis\.cache\huggingface\datasets\stanfordnlp___imdb\plain_text\0.0.0\e6281661ce1c48d982bc483cf8a173c1bbeb5d31 (last modified on Tue Jun  2 13:56:34 2026).


In [11]:
# load_dataset("stanfordnlp/sst2")

In [12]:
df = df.dropna()              # clean missing values
df = df.sample(1000, random_state=42)  # take 1k random rows

print(df.shape)
print(df.head())

(1000, 2)
                                                    text  label
6868   Dumb is as dumb does, in this thoroughly unint...      0
24016  I dug out from my garage some old musicals and...      1
9668   After watching this movie I was honestly disap...      0
13640  This movie was nominated for best picture but ...      1
14018  Just like Al Gore shook us up with his painful...      1


## Load BERT tokenizer and BERT model

In [13]:
from transformers import BertTokenizer, BertForSequenceClassification

model_name = "bert-base-uncased"

tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6058.98it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

In [14]:
def tokenize(batch):
    return tokenizer(batch["text"], padding=True, truncation=True)

tokenized_data = dataset.map(tokenize, batched=True)

In [23]:
print(tokenized_data)

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 50000
    })
})


In [24]:
print(tokenized_data["train"][0])

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [15]:
# example = tokenized_data["train"][0]
# from pprint import pprint

# pprint(example)

### View it in a readable formatted way

In [17]:
# for k, v in example.items():
#     print("\n", k)
#     print(v[:50] if isinstance(v, list) else v)
#     print("-" * 50)

## Training

In [18]:
from transformers import TrainingArguments, Trainer

In [19]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01
)

In [20]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"].shuffle().select(range(2000)),
    eval_dataset=tokenized_data["test"].select(range(500)),
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,0.406817
2,0.280100,0.293575


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.69it/s]


TrainOutput(global_step=500, training_loss=0.28009963989257813, metrics={'train_runtime': 82.4542, 'train_samples_per_second': 48.512, 'train_steps_per_second': 6.064, 'total_flos': 1052444221440000.0, 'train_loss': 0.28009963989257813, 'epoch': 2.0})

In [21]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch
0.280100,0.293575,2


{'eval_loss': 0.293574720621109}

## Inferences

In [22]:
text = "This shop is terrible!"

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

outputs = model(**inputs)
logits = outputs.logits

prediction = logits.argmax(dim=-1)
print(prediction)

tensor([0], device='cuda:0')
